In [12]:
k = 16
dmodel = 64 * k
print(f'dmodel: {dmodel}')
n_layers = k
tt_params = 12 * n_layers * (dmodel ** 2)
emb_params = 50_000 * dmodel
model_params = emb_params + tt_params

dmodel: 1024


In [18]:
batch_size = 512
seq_len = 512
steps = 20_000

tokens = batch_size * seq_len * steps
print(tokens / 1e9)

print(f'tokens/params ratio:{tokens/model_params:.2f}')

5.24288
tokens/params ratio:20.76


In [ ]:
gpus = 4
time = 23 * (60 ** 2)
# gpu_flops = 835 * 1e12  #H100
gpu_flops = 312 * 1e12  #A100

In [9]:
flops = model_params * tokens
mfu = flops / (time * gpus * gpu_flops)
print(f'FLOPS: {mfu:.2f}')

FLOPS: 0.12


In [1]:
current_steps = 370
current_time = 7 * 60
total_steps = 60_000
time_left = (total_steps / current_steps) * current_time
print(f'time left: {time_left / (60 **2 ):.2f}')

time left: 18.92


In [3]:
def calc_grid_flops(n_lrs_per_dmodel: int, dmodels: list, n_layers: int, tokens: int, n_plots: int):
    total_flops = 0
    flops_list = []
    for dmodel in dmodels:
        params = 12 * (dmodel ** 2) * n_layers
        flops = tokens * params * n_lrs_per_dmodel
        flops_list.append(flops)
        total_flops += flops
    
    return total_flops * n_plots

In [22]:
grid_dense = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128, 256, 512, 768, 1024, 1536],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 2
}

grid_moe_new = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128, 256, 512, 768, 1024, 1536, 2048, 3072],
    "n_layers": 16,
    "tokens": 5 * 1e9,
    "n_plots": 3,
}

current_128 = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 3,
}
current_256 = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [256],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 3,
}
current_512 = {
    "n_lrs_per_dmodel": 4,
    "dmodels": [512],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 1,
}
current_1024 = {
    "n_lrs_per_dmodel": 1,
    "dmodels": [1024],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 1,
}


dense_flops = calc_grid_flops(**grid_dense)
print(f"{dense_flops / 1e18} * 1e18")

moe_flops = calc_grid_flops(**grid_moe_new)
print(f"{moe_flops / 1e18} * 1e18")

cuttent_flops = calc_grid_flops(**current_128)
cuttent_flops += calc_grid_flops(**current_256)
cuttent_flops += calc_grid_flops(**current_512)
cuttent_flops += calc_grid_flops(**current_1024)
print(f"{cuttent_flops / 1e18} * 1e18")


200.0683008 * 1e18
258.8147712 * 1e18
15.325986816 * 1e18


In [11]:
16 * 64

1024